In [2]:
# Setup und Variablenindex

from pathlib import Path
import pandas as pd
import pyreadstat

data_dir = Path("data")
kohorte = "SC1"

var2pfad = {}     # Variable -> erste Datei, in der sie vorkommt
var2label = {}    # Variable -> Variablenlabel
var2werte = {}    # Variable -> Wertelabels

for f in sorted(data_dir.rglob(f"{kohorte}_NEPScomp_*.sav")):
    _, meta = pyreadstat.read_sav(f, metadataonly=True)
    for v, lab in zip(meta.column_names, meta.column_labels):
        if v not in var2pfad:
            var2pfad[v] = f
            var2label[v] = lab
            var2werte[v] = meta.variable_value_labels.get(v, {})

print(len(var2pfad), "Variablen in", kohorte)

2543 Variablen in SC1


In [ ]:
# Variablen und Wellen festlegen, die wir für die Analyse benötigen. 
# Die Zahl gibt an, in welcher Welle die Variable erhoben wurde.
# Daten laden und zusammenführen, so dass jede Zeile eine Person ist.

spezifikation = {
    # Outcome (Welle 12)
    "p724111": 12,      # Note Halbjahreszeugnis Deutsch
    "p724112": 12,      # Note Halbjahreszeugnis Mathematik
    # SES der Eltern (Welle 1)
    "p731802_g1": 1,    # Höchster Bildungsabschluss Befragte/r (ISCED)
    "p731852_g1": 1,    # Höchster Bildungsabschluss Partner/in (ISCED)
    "p731904_g14": 1,   # Berufsstatus Befragte/r (ISEI-08)
    "p731954_g14": 1,   # Berufsstatus Partner/in (ISEI-08)
    "p510006_g1": 1,    # Haushalts-Nettoeinkommen (Klassen 1-9)
    "p741001": 1,       # Haushaltsgröße
    # Kontrollvariablen (Welle 1)
    "p700010": 1,       # Geschlecht Zielkind
    "p400500_g3": 1,    # Herkunftsgruppe
    "p751001_g2": 1,    # Bundesland
    "p731110": 1,       # Familienstand Befragte/r
}

fehlend = [v for v in spezifikation if v not in var2pfad]
if fehlend:
    raise ValueError(f"Nicht in {kohorte} gefunden: {fehlend}")

def lade_var(var, welle):
    df, _ = pyreadstat.read_sav(var2pfad[var], usecols=["ID_t", "wave", var])
    df = df[df["wave"] == welle].drop(columns="wave")
    return df.set_index("ID_t")

df = pd.concat([lade_var(v, w) for v, w in spezifikation.items()], axis=1).reset_index()
print(df.shape)

(3481, 13)


In [4]:
# Prüfen 

gueltig = df.drop(columns="ID_t").notna().sum().rename("gueltige_werte").to_frame()
gueltig["anteil"] = (gueltig["gueltige_werte"] / len(df)).round(3)
display(gueltig)

for v in ["p731802_g1", "p731852_g1", "p731110", "p400500_g3", "p751001_g2", "p700010"]:
    print(v, "-", var2label[v])
    print(var2werte[v], "\n")

,gueltige_werte,anteil
p724111,1653,0.475
p724112,1655,0.475
p731802_g1,3472,0.997
p731852_g1,3224,0.926
p731904_g14,3142,0.903
p731954_g14,2981,0.856
p510006_g1,3093,0.889
p741001,3480,1.000
p700010,3481,1.000
p400500_g3,3481,1.000


p731802_g1 - Höchster Bildungsabschluss Befragter (ISCED)
{-55.0: 'nicht ermittelbar', -54.0: 'designbedingt fehlend', -20.0: 'kein Schulabschluss', 0.0: '[0A/1A] kein Abschluss', 1.0: '[2B] Haupt-, Volksschulabschluss, Berufsvorbereitende Maßnahme', 2.0: '[2A] Mittlere Reife, Realschulabschluss', 3.0: '[3A] Fachhochschulreife, Hochschulreife', 4.0: '[3B] Lehre, Berufsfachsch., Fachsch. des Gesundheitswesens (weniger als zwei Jahre), Beamter einfacher Dienst, berufliche Grundkenntnisse', 5.0: '[3C] Beamter mittlerer Dienst', 6.0: '[4A] Fachhochschulreife, Hochschulreife (second cycle)', 7.0: '[4B] Lehre, Berufsfachschule, Fachschule des Gesundheitswesens (weniger als zwei Jahre), Beamter einfacher Dienst, berufliche Grundkenntnisse (second cycle)', 8.0: '[5B] Fach- und Berufsakad. Abschl., Verwaltungsfachhochsch., Fachsch. des Gesundheitswesens (mind. zwei Jahre), Meister/Techniker, anderer Fachschulabschl., Beamter gehobener Dienst', 9.0: '[5A] Bachelor, Master, Diplom, Magister, Staa

In [ ]:
## Hier kann man nach Variablen und Labels suchen, z. B. nach "Abschluss" in den Angaben der Eltern


# Prüfen, ob beide ISCED-Variablen dieselben Codes haben
print("Gleiche Codes bei Befragte/r und Partner/in:",
      set(var2werte["p731802_g1"]) == set(var2werte["p731852_g1"]))

# Outcome: Mittel aus Deutsch- und Mathenote, nur wenn beide vorhanden sind
df["note_mittel"] = df[["p724111", "p724112"]].mean(axis=1, skipna=False)   # 1 = sehr gut
df["note_gut"] = 7 - df["note_mittel"]                                      # höher = besser

# SES Bildung: ISCED-Codes zu 4 geordneten Stufen, dann Maximum beider Eltern
stufe = {0: 1, 1: 1,
         2: 2, 4: 2, 5: 2,
         3: 3, 6: 3, 7: 3,
         8: 4, 9: 4, 10: 4}
df["bildung_befragte"] = df["p731802_g1"].map(stufe)
df["bildung_partner"] = df["p731852_g1"].map(stufe)
df["bildung_eltern"] = df[["bildung_befragte", "bildung_partner"]].max(axis=1)

print(df["bildung_eltern"].value_counts(dropna=False).sort_index())

# SES Beruf und Einkommen
df["hisei"] = df[["p731904_g14", "p731954_g14"]].max(axis=1)
df["einkommen_klasse"] = df["p510006_g1"]

# Optional: grobes Äquivalenzeinkommen aus Klassenmitten (7000 = Annahme für "6.000 und mehr")
mitte = {1: 500, 2: 1250, 3: 1750, 4: 2250, 5: 2750, 6: 3500, 7: 4500, 8: 5500, 9: 7000}
df["aeq_einkommen"] = df["p510006_g1"].map(mitte) / df["p741001"] ** 0.5

# Beschriftete Spalten für die Deskription
for v in ["p700010", "p400500_g3", "p751001_g2", "p731110"]:
    df[v + "_lbl"] = df[v].map(var2werte[v])

# Ausfall-Check: Haben Familien mit und ohne Note einen anderen SES?
df["hat_note"] = df["note_mittel"].notna()
print(df["hat_note"].value_counts())
print(df.groupby("hat_note")[["bildung_eltern", "hisei", "einkommen_klasse"]].mean().round(2))
df["bildung_3"] = df["bildung_eltern"].replace({1: 2})     # 2 = höchstens Mittlere Reife/Berufsausbildung
print(pd.crosstab(df["bildung_3"], df["hat_note"], margins=True))

# Alle IDs als Integer speichern
df["ID_t"] = df["ID_t"].astype(int)

# Speichern
Path("docs").mkdir(exist_ok=True)
doku = pd.DataFrame({
    "variable": list(spezifikation),
    "label": [var2label[v] for v in spezifikation],
    "welle": list(spezifikation.values()),
    "gueltige_werte": [int(df[v].notna().sum()) for v in spezifikation],
})
doku.to_csv("docs/variablen_SC1_auto.csv", index=False)
df.to_csv(f"data/analysedaten_{kohorte}.csv", index=False)
display(doku)

Gleiche Codes bei Befragte/r und Partner/in: True
bildung_eltern
1.0     139
2.0     633
3.0     554
4.0    2152
NaN       3
Name: count, dtype: int64
hat_note
False    1830
True     1651
Name: count, dtype: int64
          bildung_eltern  hisei  einkommen_klasse
hat_note                                         
False               3.12  54.60              5.11
True                3.62  66.51              6.09
hat_note   False  True   All
bildung_3                   
2.0          579   193   772
3.0          330   224   554
4.0          918  1234  2152
All         1827  1651  3478


,variable,label,welle,gueltige_werte
0,p724111,Note Halbjahreszeugnis Deutsch,12,1653
1,p724112,Note Halbjahreszeugnis Mathematik,12,1655
2,p731802_g1,Höchster Bildungsabschluss Befragter (ISCED),1,3472
3,p731852_g1,Höchster Bildungsabschluss Partner (ISCED),1,3224
4,p731904_g14,Berufliche Tätigkeit Befragte/r (ISEI-08),1,3142
5,p731954_g14,Berufliche Tätigkeit Partner/in (ISEI-08),1,2981
6,p510006_g1,Haushalts-Nettoeinkommen (inkl. kategorisierte...,1,3093
7,p741001,Haushaltsgröße,1,3480
8,p700010,Geschlecht Zielkind,1,3481
9,p400500_g3,Herkunftsgruppe,1,3481
